In [ ]:
import numpy as np
import pandas as pd
from tabulate import tabulate

In [ ]:
truth = pd.read_parquet("../data/processed/evaluation.parquet")

source = globals().get("source", "desc")

predictions = globals().get(
    "predictions",
    {"subheading": [[c] for c in truth["subheading"].str[0]]},
)

In [ ]:
def rank_match(y, y_hat):
    for i, code in enumerate(y_hat, 1):
        if code in y:
            return i
    return np.inf


def evaluate_predictions(truth, prediction):
    flag = np.ones(len(truth), bool)
    rank = np.array([
        rank_match(y, y_hat) for y, y_hat in zip(truth, prediction)
    ])[flag]

    return pd.DataFrame([{
        "R@1": (rank <= 1).mean(),
        "R@5": (rank <= 5).mean(),
        "R@10": (rank <= 10).mean(),
        "MRR": (1 / rank).mean(),
    }]).round(3)


data = []
for tag, prediction in predictions.items():
    metric = evaluate_predictions(truth[tag], prediction)
    metric.index = [tag]
    data.append(metric)
metrics = pd.concat(data)

print(f"Evaluation for HS ({source}) (n={len(truth)})")
print(tabulate(metrics, headers="keys", tablefmt="psql"))